<a href="https://colab.research.google.com/github/EdithGoren/email_finetune_project/blob/feature-emails/email_finetune_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv('exe.csv')  # טוען את הקובץ לתוך משתנה בשם df
print(df.head())             # מציג את 5 השורות הראשונות

                         input  \
0         cold meeting request   
1    follow-up meeting request   
2         team meeting request   
3    meeting request with boss   
4  client presentation meeting   

                                              output  
0  Subject: Brief meeting about [specific topic]\...  
1  Subject: Follow-up Discussion: [Previous Meeti...  
2  Subject: Team Sync-Up: Q4 Planning\n\nHi Team,...  
3  Subject: Quick Check-in Request\n\nHi [Boss's ...  
4  Subject: [Company] Solution Presentation\n\nDe...  


In [3]:
!pip install transformers datasets --quiet


In [4]:
!pip install accelerate --quiet

In [5]:
import pandas as pd

# טען את הקובץ שהעלית (אם שמרת בשם אחר, שנה כאן)
df = pd.read_csv('exe.csv')

# הצג את 5 השורות הראשונות לבדיקה
print(df.head())

                         input  \
0         cold meeting request   
1    follow-up meeting request   
2         team meeting request   
3    meeting request with boss   
4  client presentation meeting   

                                              output  
0  Subject: Brief meeting about [specific topic]\...  
1  Subject: Follow-up Discussion: [Previous Meeti...  
2  Subject: Team Sync-Up: Q4 Planning\n\nHi Team,...  
3  Subject: Quick Check-in Request\n\nHi [Boss's ...  
4  Subject: [Company] Solution Presentation\n\nDe...  


In [6]:
from datasets import Dataset

# המרה ל-Dataset של Hugging Face
dataset = Dataset.from_pandas(df)

# בדיקה
print(dataset[0])

{'input': 'cold meeting request', 'output': "Subject: Brief meeting about [specific topic]\\n\\nHi [Name],\\n\\nI'm [Your Name] from [Company], and I noticed your recent work on [specific project/achievement]. I'd love to schedule a brief meeting to discuss how we might collaborate on [specific opportunity].\\n\\nWould you have 15 minutes next week? I'm available on:\\n- Tuesday at 10 AM\\n- Wednesday at 2 PM\\n- Thursday at 11 AM\\n\\nYou can schedule directly on my calendar: [zeeg.me/your-name].\\n\\nLooking forward to your response.\\n\\nBest regards,\\n[Your Name]"}


In [7]:
from datasets import DatasetDict

# 90% לאימון, 10% לבדיקה
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 22
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 3
    })
})


In [8]:
from transformers import AutoTokenizer

model_checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# הוספת שורת תיקון:
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    return tokenizer(
        examples["input"],
        text_target=examples["output"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_dataset = dataset.map(preprocess_function, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [13]:
!pip install --upgrade transformers

In [14]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

model_checkpoint = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

# Data collator - דואג לריפוד אוטומטי
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # לא מסכת מילים, כי זה מודל גנרטיבי
)

In [10]:
!pip install --upgrade transformers

In [24]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=10,
    push_to_hub=False,  # אם אין לך טוקן של HF
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipython-input-25-1336553227.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.852000,3.909362
2,2.143600,3.684814
3,1.943800,3.702264


TrainOutput(global_step=33, training_loss=1.8899251186486445, metrics={'train_runtime': 18.7053, 'train_samples_per_second': 3.528, 'train_steps_per_second': 1.764, 'total_flos': 4311318528000.0, 'train_loss': 1.8899251186486445, 'epoch': 3.0})

In [27]:
trainer.save_model("finetuned-gpt2-emails")
tokenizer.save_pretrained("finetuned-gpt2-emails")

('finetuned-gpt2-emails/tokenizer_config.json',
 'finetuned-gpt2-emails/special_tokens_map.json',
 'finetuned-gpt2-emails/vocab.json',
 'finetuned-gpt2-emails/merges.txt',
 'finetuned-gpt2-emails/added_tokens.json',
 'finetuned-gpt2-emails/tokenizer.json')

In [28]:
from transformers import pipeline

pipe = pipeline("text-generation", model="finetuned-gpt2-emails", tokenizer="finetuned-gpt2-emails")

prompt = "cold meeting request"
result = pipe(prompt, max_new_tokens=150)
print(result[0]['generated_text'])

Device set to use cuda:0


cold meeting request in their meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting request meeting re

In [29]:
prompt = "input: cold meeting request\noutput:"
result = pipe(prompt, max_new_tokens=200)
print(result[0]['generated_text'])

input: cold meeting request
output: cold meeting request
<!DOCTYPE_VERSION> /// Create a new cold meeting request request to be used later. /// /// @param request type request /// @param request_type request type /// @return request request request key /// @see #shared_request_key public struct Request { /// Request type and request key for the request request key /// request.request.request /// Request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request request r

In [30]:
prompt = "input: cold meeting request\noutput:"
result = pipe(prompt, max_new_tokens=200, do_sample=True, temperature=0.8, top_p=0.95)
print(result[0]['generated_text'])

input: cold meeting request
output: cold meeting request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request
contact request

In [33]:
   trainer.save_model("finetuned-gpt2-emails")

In [34]:
   import os
   print(os.listdir())
   print(os.listdir('finetuned-gpt2-emails'))

['.config', 'finetuned-gpt2-emails', 'drive', 'exe.csv', 'results', 'sample_data']
['vocab.json', 'merges.txt', 'special_tokens_map.json', 'training_args.bin', 'tokenizer.json', 'model.safetensors', 'tokenizer_config.json', 'config.json', 'generation_config.json']


In [35]:
from google.colab import files
files.download('finetuned-gpt2-emails/pytorch_model.bin')

FileNotFoundError: Cannot find file: finetuned-gpt2-emails/pytorch_model.bin

In [36]:
from google.colab import files
files.download('finetuned-gpt2-emails/model.safetensors')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>